# 🏆 Capstone Step 4 — Structured Output & Guardrails (M2 + M7)
**Prerequisite:** Complete `lesson_09c_agent_orchestrator.ipynb` first

In [ ]:
# ── Capstone imports & env check ──────────────────────────────────
import os, json, time, datetime, re, asyncio
from pathlib import Path
from typing import Optional, List, Any
from pydantic import BaseModel, Field, field_validator
from dataclasses import dataclass, field as dc_field
import pandas as pd
import numpy as np

from openai import OpenAI
from anthropic import Anthropic
from datasets import load_dataset
from llm_checker import check, hint, evaluate, progress, show_exercise

openai_client = OpenAI()
claude_client = Anthropic()
lm_client     = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")

# Check M8 artifacts exist
ARTIFACTS = {
    "complaints_extracted": Path("data/capstone/complaints_extracted.jsonl"),
    "hybrid_routing_log":   Path("data/capstone/hybrid_routing_log.jsonl"),
    "golden_eval":          Path("data/capstone/golden_eval.jsonl"),
}
for name, path in ARTIFACTS.items():
    status = "✅" if path.exists() else "⚠️  MISSING — run Module 8 first"
    print(f"  {status}  {name}: {path}")

print("\n✅ Capstone imports ready")

---
## Step 4 — Structured Output & Guardrails (M2 + M7)
**Goal:** Generate 100 `CRMIntelligenceReport` objects with full Pydantic validation, HITL interrupt for offers > 100 PLN.

**Auto-check verifies:**
- ✓ `CRMIntelligenceReport` validates without error for 100 customers
- ✓ Human approval triggered for offers > 100 PLN
- ✓ `total_cost_usd` tracked per report


In [ ]:
show_exercise(
    "CAP-4", "Structured Output & Guardrails",
    "Generate CRMIntelligenceReport for 100 customers. All fields Pydantic-validated. "
    "HITL interrupt for offer_value > 100 PLN. Track cost per report.",
    hints=[
        "▸ All fields have Pydantic types and constraints",
        "▸ processing_time_ms measured with time.perf_counter()",
        "▸ Loop over INGESTED_RECORDS — use complaint as input for each customer",
    ],
    checks=[
        "CRMIntelligenceReport validates for 100 customers",
        "Human approval triggered for offers > 100 PLN",
        "total_cost_usd tracked per report"
    ]
)

In [ ]:
# ── Step 4: Generate 100 CRMIntelligenceReport objects ────────────
ALL_REPORTS: List[CRMIntelligenceReport] = []
cap4_failures = []
HITL_TRIGGERED = []

# Use first 100 ingested records (or generate dummy ones if fewer)
records_to_process = INGESTED_RECORDS[:100]
if len(records_to_process) < 100:
    print(f"⚠️  Only {len(records_to_process)} records — processing all available")

for i, rec in enumerate(records_to_process):
    t0 = time.perf_counter()
    try:
        cid = rec.customer_id
        # Run pipeline
        pipeline_state = run_capstone_pipeline(
            cid,
            f"Process complaint: {rec.summary}",
            complaint=rec,
            auto_approve=False  # respect HITL
        )
        churn = pipeline_state.get("churn_risk") or {}
        offer_val = pipeline_state.get("offer_value_pln", 0.0)

        # HITL check
        if offer_val > 100:
            HITL_TRIGGERED.append(cid)
            # In production: pause here for human review
            # For notebook: auto-approve but flag it
            pipeline_state = cap_app.invoke(None, {"configurable":{"thread_id":cid}})

        elapsed_ms = int((time.perf_counter() - t0) * 1000)

        # Build structured report
        offer_obj = None
        if pipeline_state.get("offer_draft") and not pipeline_state.get("offer_blocked"):
            offer_obj = RetentionOffer(
                customer_id=cid,
                offer_text=pipeline_state["offer_draft"],
                offer_value_pln=offer_val,
                approved=cid not in HITL_TRIGGERED
            )

        report = CRMIntelligenceReport(
            customer_id=cid,
            complaint=rec,
            churn_risk=ChurnRiskCard(
                customer_id=cid,
                churn_proba=churn.get("churn_proba", 0.0),
                risk_tier=churn.get("risk_tier", "unknown"),
                top_features=churn.get("top_features", []),
            ),
            policy_match=PolicySnippet(
                query=pipeline_state.get("query",""),
                content=pipeline_state.get("policy_snippet","")[:500],
            ),
            retention_offer=offer_obj,
            audit_trail=[AuditEntry(**e) for e in pipeline_state.get("audit_log",[])],
            processing_time_ms=elapsed_ms,
            total_cost_usd=round(pipeline_state.get("total_cost_usd", 0.0), 6),
        )
        ALL_REPORTS.append(report)

    except Exception as e:
        cap4_failures.append({"customer_id": rec.customer_id, "error": str(e)})

    if (i + 1) % 10 == 0:
        print(f"  Processed {i+1}/{len(records_to_process)} customers ...")

print(f"\n✅ Step 4 complete: {len(ALL_REPORTS)} reports | {len(cap4_failures)} failures")
print(f"   HITL triggered: {len(HITL_TRIGGERED)} customers (offer > 100 PLN)")
print(f"   Total cost: ${sum(r.total_cost_usd for r in ALL_REPORTS):.4f}")

In [ ]:
# ── Auto-check Step 4 ─────────────────────────────────────────────
validation_errors = 0
for r in ALL_REPORTS:
    try:
        CRMIntelligenceReport.model_validate(r.model_dump())
    except Exception:
        validation_errors += 1

total_cost_tracked = all(r.total_cost_usd >= 0 for r in ALL_REPORTS)

check([
    (len(ALL_REPORTS) >= 90,            f"≥ 90 reports generated (got {len(ALL_REPORTS)})"),
    (validation_errors == 0,            "All reports pass Pydantic validation"),
    (isinstance(HITL_TRIGGERED, list),  "HITL trigger list exists"),
    (total_cost_tracked,                "total_cost_usd tracked in every report"),
    (all(len(r.audit_trail) >= 1 for r in ALL_REPORTS), "Every report has ≥ 1 audit entry"),
], exercise_id="CAP-4")

---
## ✅ Step 4 Complete
- ☐ CRMIntelligenceReport validates for 100 customers
- ☐ Human approval triggered for offers > 100 PLN
- ☐ total_cost_usd tracked per report

➡️ Continue to `lesson_09e_eval_monitoring.ipynb`